# 41 · Is Ridge’s edge robust to influential stocks, risk exposure and tail events?

**Development-only forensic research.** Refit the frozen Ridge α=100 baseline once per chronological fold, then reuse its *ex-ante* validation scores across many diagnostics. **No final historical holdout, no prospective prediction ledger.** Transaction costs remain zero. A 2:1 win/loss payoff ratio is measured, not assumed. The model has been chosen after earlier exploration, so these results remain exploratory.

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(),f'Run from repository root or notebooks/, not {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.edge_robustness_41 import *
OUT=ROOT/'artifacts/edge_robustness/1d';OUT.mkdir(parents=True,exist_ok=True)
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'Inference rows:',latest.shape,'Output:',OUT)

## Experiments 1–3 — Reconstruct the closed-holdout-safe universe and chronological folds

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,features=add_volume_regime_features(expanded)
print('Development:',expanded.shape,'Feature extras:',len(features),'Holdout cutoff:',cutoff)
assert len(expanded)>0 and len(features)>0

## Experiments 4–7 — Frozen Ridge α100 refit, chronological leakage audit, coefficient drift and score distributions

In [ ]:
start=time.perf_counter()
scored,coef,audit=fit_and_score(expanded,features,cap=60000,seed=42)
scored=score_variants(scored)
print('Training and scoring seconds:',round(time.perf_counter()-start,1))
display(audit)
coef_pivot=coef.pivot(index='feature',columns='fold',values='coef')
coef_pivot['mean_abs']=coef_pivot[[1,2,3]].abs().mean(axis=1)
coef_pivot['same_sign']=((coef_pivot[[1,2,3]]>0).all(axis=1)|(coef_pivot[[1,2,3]]<0).all(axis=1))
display(coef_pivot.sort_values('mean_abs',ascending=False).head(30).round(6))
print('Coefficient sign stability:',coef_pivot.same_sign.mean())

## Experiments 8–25 — Fully reranked leave-one-symbol-out, top-contributor exclusion, signal thresholds and negative controls

**Important:** Exclusions are applied *before* ranking each date, so the portfolio refills from eligible stocks. This tests exclusion of symbols at selection time; it does **not** retrain Ridge without those symbols. The list of influential stocks was identified using earlier validation results and is therefore exploratory. Score percentile and z-score are rank-preserving negative controls.

In [ ]:
results=run_portfolio_stress(scored)
metrics,daily,trades=results['metrics'],results['daily'],results['trades']
leader=metrics.groupby(['model','k'],as_index=False).agg(mean_return=('mean_daily','mean'),market_excess=('market_excess','mean'),payoff=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),positive_folds=('mean_daily',lambda z:int((z>0).sum())),worst_drawdown=('max_drawdown','min'))
display(leader.loc[leader['k'].eq(5)].sort_values('mean_return',ascending=False).round(6))
print('Fold-level five-position returns:')
display(metrics.loc[metrics['k'].eq(5)].pivot(index='model',columns='fold',values='mean_daily').round(6))

## Experiments 26–29 — Position count, exposure and symbol concentration

In [ ]:
display(leader.sort_values(['k','mean_return'],ascending=[True,False]).round(6))
exposure=exposure_diagnostics(trades,scored)
print('Largest cumulative return contributors:');display(exposure['symbol'].head(30).round(6))
print('Most frequent picks by fold:');display(exposure['frequency'].sort_values('share_of_days',ascending=False).head(30).round(5))
print('Selected-stock observable-factor percentiles:');display(exposure['factor'].round(5))

## Experiments 30–33 — Exceptional winners, data-integrity flags, unusual returns and return concentration

These are **diagnostic** stress tests. Outlier trades are not removed from performance estimates based on their realized outcomes. Audit corporate actions, splits, price adjustment and symbol history externally before interpreting extreme bars as genuine trading opportunities.

In [ ]:
integrity,outliers,concentration=tail_and_integrity(scored,trades)
display(integrity);display(outliers[['fold','bar_timestamp','symbol','score','next_return','rank']].head(40));display(concentration.round(5))
print('Extreme return prevalence by fold:')
display(scored.assign(abs_gt20=scored.next_return.abs()>.2).groupby('fold').agg(rows=('symbol','size'),gt20=('abs_gt20','sum'),min_return=('next_return','min'),max_return=('next_return','max')).round(5))

## Experiments 34–36 — Daily rank IC, score spread, and market-relative attribution

In [ ]:
quality=score_quality(scored)
display(quality.groupby('fold').agg(mean_ic=('ic','mean'),median_ic=('ic','median'),positive_ic_days=('ic',lambda x:float((x>0).mean())),mean_score_spread=('score_spread','mean'),mean_return_spread=('return_spread','mean')).round(6))
base=metrics.loc[metrics['model'].eq('ridge_all')];display(base[['fold','k','mean_daily','market_excess','win_rate','avg_win','avg_loss','payoff_ratio','profit_factor','max_drawdown']].round(6))
fig,ax=plt.subplots(figsize=(10,4))
for fold,g in quality.groupby('fold'):
    ax.plot(pd.to_datetime(g.bar_timestamp),g.ic.rolling(30,min_periods=10).mean(),label=f'Fold {fold}')
ax.axhline(0,linestyle='--',linewidth=1);ax.legend();ax.set_title('30-day rolling cross-sectional rank IC');plt.show()

## Experiments 37–39 — Paired 10-day-block bootstrap, chronology and robustness criteria

Multiple configurations are explored. Individual bootstrap confidence intervals are **not** multiple-testing-adjusted. Exclusion configurations are diagnostic and not confirmatory because their names were selected from previous validation outcomes.

In [ ]:
paired=paired_inference(daily,reps=1200)
display(paired.loc[paired['model'].isin(['exclude_CVNA','exclude_PLTR','exclude_CVNA_PLTR','exclude_top5_contributors']) & paired['k'].eq(5)].sort_values(['model','fold']).round(6))
summary=paired.groupby('model',as_index=False).agg(mean_lift=('mean_lift','mean'),positive_folds=('mean_lift',lambda v:int((v>0).sum())),ci_above_zero=('ci_low',lambda v:int((v>0).sum())))
display(summary.sort_values('mean_lift',ascending=False).round(6))

## Experiments 40–43 — Risk of a misleading edge: top contributors, 2:1 payoff, drawdowns and control checks

In [ ]:
baseline=metrics.loc[metrics['model'].eq('ridge_all') & metrics['k'].eq(5)].sort_values('fold')
for model in ('exclude_CVNA','exclude_PLTR','exclude_CVNA_PLTR','exclude_top5_contributors'):
    comp=metrics.loc[metrics['model'].eq(model) & metrics['k'].eq(5)].sort_values('fold')
    print(model,'average lift vs baseline:',round(float((comp.mean_daily.to_numpy()-baseline.mean_daily.to_numpy()).mean()),6),'fold-positive:',int((comp.mean_daily.to_numpy()>0).sum()))
for name in ('rank_pct','score_z'):
    a=trades.loc[trades['model'].eq('ridge_all') & trades['k'].eq(5)][['fold','bar_timestamp','symbol']]
    b=trades.loc[trades['model'].eq(name) & trades['k'].eq(5)][['fold','bar_timestamp','symbol']]
    print('Ranking invariant control',name, 'identical selections:',set(map(tuple,a.to_numpy()))==set(map(tuple,b.to_numpy())))
print('2:1 achieved on each fold?',baseline[['fold','payoff_ratio','meets_2_to_1']].to_string(index=False))

## Experiment 44 — Persist reproducible outputs and determine what remains unproven

In [ ]:
save_outputs(OUT,scored=scored,coefficients=coef,training_audit=audit,metrics=metrics,daily=daily,trades=trades,leaderboard=leader,paired=paired,paired_summary=summary,concentration=concentration,integrity=integrity,extreme_trades=outliers,score_quality=quality,symbol_exposure=exposure['symbol'],factor_exposure=exposure['factor'])
print('Saved files to',OUT)
print('INTERPRETATION: Development-only; previous exploration informed baseline and excluded symbols. Do not claim statistical proof, deployment readiness or 2:1 payoff without evidence.')
print('Remaining limitations: point-in-time sector history, delisted names, corporate-action audit, true retraining leave-symbol-out, untouched final holdout and prospective trading.')